In [ ]:
import pandas as pd

df_full = pd.read_csv('df_with_marking_final_full.csv')

In [2]:
df_marking = df_full.sample(n=300, random_state=42).reset_index(drop=True)

In [ ]:
import re
import pandas as pd

def get_full_text(row):
    return f"{str(row['preamble'])} {str(row['description'])} {str(row['sentence'])}"

alcohol_keywords = {
    'да': [
        r"в состоянии\s+алкогольного\s+опьянения",
        r"алкогольное\s+опьянение",
        r"находясь\s+в\s+состоянии\s+опьянения",
        r"находился\s+в\s+состоянии\s+алкогольного\s+опьянения",
        r"выпив\s+(алкоголь|спиртное)",
        r"употребив\s+(алкоголь|спиртное)",
        r"был\s+пьян", r"была\s+пьяна",
        r"выпивал\s+(водку|пиво|спиртное)",
        r"влиянием\s+алкоголя",
        r"опьянение,\s+вызванное\s+употреблением\s+алкоголя"
    ],
    'нет': [] 
}

train_data_alcohol = []
s = 0

for idx, row in df_marking.iterrows():
    text = get_full_text(row)
    text_lower = text.lower()

    true_label = str(row.get("predicted_alcohol")).strip().lower()
    if true_label not in alcohol_keywords:
        continue

    found = False
    for pattern in alcohol_keywords['да']:  
        match = re.search(pattern, text_lower)
        if match:
            start, end = match.span()
            train_data_alcohol.append((text, {"entities": [(start, end, "ALCOHOL")]}))
            found = True
            break

    if not found:
        train_data_alcohol.append((text, {"entities": []}))
        print(f"Не найдены ключевые слова 'да' в id={idx}")
        s += 1

print(f"Не найдены ключевые слова в {s} примерах")
print(f"TRAIN_DATA_ALCOHOL готово: {len(train_data_alcohol)} примеров")

Не найдены ключевые слова 'да' в id=0
Не найдены ключевые слова 'да' в id=3
Не найдены ключевые слова 'да' в id=4
Не найдены ключевые слова 'да' в id=7
Не найдены ключевые слова 'да' в id=10
Не найдены ключевые слова 'да' в id=12
Не найдены ключевые слова 'да' в id=41
Не найдены ключевые слова 'да' в id=43
Не найдены ключевые слова 'да' в id=46
Не найдены ключевые слова 'да' в id=62
Не найдены ключевые слова 'да' в id=63
Не найдены ключевые слова 'да' в id=64
Не найдены ключевые слова 'да' в id=70
Не найдены ключевые слова 'да' в id=71
Не найдены ключевые слова 'да' в id=77
Не найдены ключевые слова 'да' в id=80
Не найдены ключевые слова 'да' в id=85
Не найдены ключевые слова 'да' в id=89
Не найдены ключевые слова 'да' в id=95
Не найдены ключевые слова 'да' в id=115
Не найдены ключевые слова 'да' в id=117
Не найдены ключевые слова 'да' в id=126
Не найдены ключевые слова 'да' в id=131
Не найдены ключевые слова 'да' в id=133
Не найдены ключевые слова 'да' в id=138
Не найдены ключевые сло

In [ ]:
import spacy
from spacy.training import Example
from spacy.util import minibatch
import random
import warnings

nlp = spacy.blank("ru")

if "ner" not in nlp.pipe_names:
    ner = nlp.add_pipe("ner")
else:
    ner = nlp.get_pipe("ner")

ner.add_label("ALCOHOL")

examples = []
for text, annot in train_data_alcohol:
    doc = nlp.make_doc(text)
    example = Example.from_dict(doc, annot)
    examples.append(example)

optimizer = nlp.begin_training()
warnings.filterwarnings("ignore")

for i in range(15):
    random.shuffle(examples)
    losses = {}
    batches = minibatch(examples, size=8)
    for batch in batches:
        nlp.update(batch, drop=0.2, losses=losses)
    print(f"Epoch {i + 1}, Losses: {losses}")

nlp.to_disk("ner_b_alcohol_model")
print("Модель сохранена в 'ner_b_alcohol_model'")

/Users/ekaterinastepura/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Epoch 1, Losses: {'ner': 245311.91073117033}
Epoch 2, Losses: {'ner': 1096.2518579027007}
Epoch 3, Losses: {'ner': 362.78070940602004}
Epoch 4, Losses: {'ner': 360.0806889992538}
Epoch 5, Losses: {'ner': 306.1276840602688}
Epoch 6, Losses: {'ner': 299.13481988387207}
Epoch 7, Losses: {'ner': 278.7756240715707}
Epoch 8, Losses: {'ner': 265.2732728649121}
Epoch 9, Losses: {'ner': 244.6441255897339}
Epoch 10, Losses: {'ner': 239.36304089429566}
Epoch 11, Losses: {'ner': 236.95993139534178}
Epoch 12, Losses: {'ner': 223.30498735449694}
Epoch 13, Losses: {'ner': 207.92799541308818}
Epoch 14, Losses: {'ner': 196.21644931995348}
Epoch 15, Losses: {'ner': 162.29403080554343}
Модель сохранена в 'ner_b_alcohol_model'


In [6]:
df_test = pd.read_csv('df_with_marking_final.csv')

In [14]:
import spacy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

nlp = spacy.load("ner_b_alcohol_model")

preds = []
true_labels = []

for idx, row in df_test.iterrows():
    text = get_full_text(row)
    true_label = str(row.get("alcohol")).strip().lower()

    if true_label not in ["да", "нет"]:
        continue

    doc = nlp(text)
    predicted_label = "да" if any(ent.label_ == "ALCOHOL" for ent in doc.ents) else "нет"

    preds.append(predicted_label)
    true_labels.append(true_label)

print("\nМетрики качества для признака 'alcohol':")
print(f"Accuracy:  {accuracy_score(true_labels, preds):.2f}")
print(f"F1-score:  {f1_score(true_labels, preds, pos_label='да'):.2f}")


Метрики качества для признака 'alcohol':
Accuracy:  0.73
F1-score:  0.81
